In [ ]:
# Install required libraries
!pip install torch rdkit pandas scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 17.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# Path to your dataset
dataset_path = "/content/drive/MyDrive/Projects/NeurIPS Polymer Prediction 2025/data/train.csv"

# Load the data
df = pd.read_csv(dataset_path)
df = df[df['SMILES'].notna() & df['Rg'].notna()] # Remove rows with missing labels for simplicity
df.head()


,id,SMILES,Tg,FFV,Tc,Density,Rg
10,2986007,*c1ccc(-c2ccc3c(c2)C(CCCCCCC#N)(CCCCCCC#N)c2cc...,NaN,0.402397,0.487000,0.901123,28.682441
11,3013292,*CC(*)c1ccc(C(=O)O)c(C(=O)O)c1,NaN,NaN,0.171000,1.184354,13.534248
37,9231170,*CC(*)CC,NaN,NaN,0.225333,NaN,13.872913
46,11894206,*CC(*)C(=O)Oc1ccccc1C,NaN,0.357831,0.173500,1.061864,12.737463
64,16855553,*CC(*)C(=O)Oc1ccccc1,NaN,0.353941,0.196000,1.109389,13.435339


In [ ]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.6 MB/s eta 0:00:00


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import torch
from torch_geometric.data import Data

def atom_features(atom):
    return torch.tensor([
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        atom.GetHybridization().real,
        atom.GetIsAromatic()
    ], dtype=torch.float)

def bond_features(bond):
    return torch.tensor([
        bond.GetBondTypeAsDouble(),
        bond.IsInRing()
    ], dtype=torch.float)

def smiles_to_graph(smiles, y):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    x = torch.stack([atom_features(atom) for atom in mol.GetAtoms()])
    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index += [[i, j], [j, i]]
        edge_attr += [bond_features(bond), bond_features(bond)]

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.stack(edge_attr)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=torch.tensor([y], dtype=torch.float))


In [ ]:
import torch.nn as nn
from torch_geometric.nn import GCNConv, global_mean_pool

class GNNRegressor(nn.Module):
    def __init__(self, node_feat_dim, hidden_dim=64):
        super().__init__()
        self.conv1 = GCNConv(node_feat_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin1 = nn.Linear(hidden_dim, 64)
        self.lin2 = nn.Linear(64, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = torch.relu(self.conv1(x, edge_index))
        x = torch.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        x = torch.relu(self.lin1(x))
        return self.lin2(x)


In [ ]:
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split

graphs = []
for _, row in df.iterrows():
    data = smiles_to_graph(row['SMILES'], row['Rg'])
    if data is not None:
        graphs.append(data)

train_graphs, test_graphs = train_test_split(graphs, test_size=0.2, random_state=42)
train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=32)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GNNRegressor(node_feat_dim=train_graphs[0].x.size(1)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

def train():
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = loss_fn(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def test():
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            pred = model(batch)
            preds.append(pred.cpu())
            truths.append(batch.y.cpu())
    return torch.cat(preds), torch.cat(truths)


In [ ]:
best_mse = float('inf')
best_model_path = "/content/drive/MyDrive/Projects/NeurIPS Polymer Prediction 2025/models/gnn_best_rg_model.pth"

for epoch in range(1000):
    loss = train()
    preds, truths = test()
    mse = torch.mean(torch.abs(preds - truths)**2).item()
    print(f"Epoch {epoch:02d} | Train Loss: {loss:.4f} | Test MSE: {mse:.4f}")
    if mse < best_mse:
        best_mse = mse
        torch.save(model.state_dict(), best_model_path)
        print(f"New best model saved (MSE: {mse:.4f})")


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([11])) that is different to the input size (torch.Size([11, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 00 | Train Loss: 267.5259 | Test MSE: 252.6464
New best model saved (MSE: 252.6464)
Epoch 01 | Train Loss: 202.9238 | Test MSE: 155.9255
New best model saved (MSE: 155.9255)
Epoch 02 | Train Loss: 82.6419 | Test MSE: 28.8861
New best model saved (MSE: 28.8861)
Epoch 03 | Train Loss: 27.0260 | Test MSE: 25.1211
New best model saved (MSE: 25.1211)
Epoch 04 | Train Loss: 23.4040 | Test MSE: 25.8396
Epoch 05 | Train Loss: 23.4688 | Test MSE: 23.9891
New best model saved (MSE: 23.9891)
Epoch 06 | Train Loss: 22.4180 | Test MSE: 24.0280
Epoch 07 | Train Loss: 22.4998 | Test MSE: 24.1777
Epoch 08 | Train Loss: 23.6424 | Test MSE: 23.9641
New best model saved (MSE: 23.9641)
Epoch 09 | Train Loss: 22.4420 | Test MSE: 24.0667
Epoch 10 | Train Loss: 22.3395 | Test MSE: 24.0062
Epoch 11 | Train Loss: 22.4025 | Test MSE: 24.0972
Epoch 12 | Train Loss: 22.3767 | Test MSE: 23.8365
New best model saved (MSE: 23.8365)
Epoch 13 | Train Loss: 22.3320 | Test MSE: 24.2869
Epoch 14 | Train Loss: 22.95